#| hide 

NBDEV: # nbdev

nbdev makes exploration an integral part of your workflow, all while promoting software engineering best practices. It allows you to write, test, document, and distribute software packages and technical articles — all in one place, your notebook.

Here are some of nbdev's features:

- **Supports literate programming**: write prose, code, and tests in notebooks — no context-switching
- **Git-friendly notebooks**: human-readable merge conflicts; no unwanted metadata
- **Two-way sync between notebooks and plaintext source code**: allows you to use your IDE for code navigation or quick edits
- **Beautiful technical documentation**: documentation is automatically generated using Quarto and hosted on GitHub Pages; docs support LaTeX, are searchable, and are automatically hyperlinked (including out-of-the-box support for many packages via nbdev-index)
- **Seamless testing**: write tests as ordinary notebook cells, run them in parallel with a single command; out-of-the-box continuous integration with GitHub Actions to ensure tests pass before every merge
- **Simplified package release**: publish packages to PyPI and conda; Python best practices are automatically followed (e.g. only exported objects are included in `__all__`)

## Tutorials

- [Getting started](https://nbdev.fast.ai/getting_started.html.md)
- [Tutorial](https://nbdev.fast.ai/tutorials/tutorial.html.md)
- [Best practices](https://nbdev.fast.ai/tutorials/best_practices.html.md)

## API

- [API List](https://nbdev.fast.ai/apilist.txt): A succint list of all functions and methods in nbdev.

## Optional

- [Git-Friendly Jupyter](https://nbdev.fast.ai/tutorials/git_friendly_jupyter.html.md): How to use nbdev hooks for git-friendly Jupyter notebooks
- [Blogging](https://nbdev.fast.ai/tutorials/blogging.html.md: Creating a blog with notebooks)
- [Pre-Commit Hooks](https://nbdev.fast.ai/tutorials/pre_commit.html.md): How to use nbdev’s git pre-commit hooks
- [Documentation Only Sites](https://nbdev.fast.ai/tutorials/docs_only.html.md): How to create nbdev powered docs without a library!
- [Modular nbdev](https://nbdev.fast.ai/tutorials/modular_nbdev.html.md): How to use nbdev’s various tools separately
- [Writing nbdev plugins](https://nbdev.fast.ai/tutorials/extensions.html.md): How to customize nbdev processors to do what you want
- [nbdev1 Migration](https://nbdev.fast.ai/tutorials/migrating.html.md): How to change your nbdev1 repo to work with nbdev2
- [nbdev.config](https://nbdev.fast.ai/api/config.html.md): Configuring nbdev and bootstrapping notebook export
- [nbdev.maker](https://nbdev.fast.ai/api/maker.html.md): Create one or more modules from selected notebook cells
- [nbdev.process](https://nbdev.fast.ai/api/process.html.md): A notebook processor
- [nbdev.export](https://nbdev.fast.ai/api/export.html.md): Exporting a notebook to a library
- [nbdev.doclinks](https://nbdev.fast.ai/api/doclinks.html.md): Generating a documentation index from a module
- [nbdev.sync](https://nbdev.fast.ai/api/sync.html.md): Propagate small changes in the library back to notebooks
- [nbdev.merge](https://nbdev.fast.ai/api/merge.html.md): Fix merge conflicts in jupyter notebooks
- [nbdev.showdoc](https://nbdev.fast.ai/api/showdoc.html.md): Display symbol documentation in notebook and website
- [nbdev.frontmatter](https://nbdev.fast.ai/api/frontmatter.html.md): A YAML and formatted-markdown frontmatter processor
- [nbdev.processor](https://nbdev.fast.ai/api/processors.html.md): Some processors for NBProcessor
- [nbdev.clean](https://nbdev.fast.ai/api/clean.html.md): Strip superfluous metadata from notebooks
- [nbdev.test](https://nbdev.fast.ai/api/test.html.md): Run unit tests on notebooks in parallel
- [nbdev.cli](https://nbdev.fast.ai/api/cli.html.md): CLI commands
- [nbdev.quarto](https://nbdev.fast.ai/api/quarto.html.md): Install and interact with Quarto from nbdev
- [nbdev.qmd](https://nbdev.fast.ai/api/qmd.html.md): Basic qmd generation helpers (experimental)
- [nbdev.migrate](https://nbdev.fast.ai/api/migrate.html.md): Utilities for migrating to nbdev
- [nbdev.serve](https://nbdev.fast.ai/api/serve.html.md): A parallel ipynb processor (experimental)
- [nbdev.release](https://nbdev.fast.ai/api/release.html.md): Auto-generated tagged releases and release notes from GitHub issues

#|hide 

pandas_gbq docs: https://pandas-gbq.readthedocs.io/en/latest/


#| hide

Curernt logic

```python
"""A wrapper for pandas_gbq to make it slightly more ergonomic with some extra convenience funcs
"""
from pandas_gbq import read_gbq as _original_read_gbq, to_gbq as _original_to_gbq, Context, context
import pandas as pd, re, time
from decimal import Decimal
from google.cloud import bigquery

__all__ = ['read', 'to', 'ex', 'Context', 'context']

def read(query_or_table, verbose=True, convert_types=True, date_cols=None, str_cols=["scv_id"], use_bqstorage_api=True, **kwargs):
    "Read from BigQuery with optional type conversion and verbose output"
    start = time.time()
    df = _original_read_gbq(query_or_table, use_bqstorage_api=use_bqstorage_api, **kwargs)
    if convert_types: df = convert_bq_dtypes(df, date_cols=date_cols, str_cols=str_cols)
    if verbose:
        elapsed = time.time() - start
        print(f"Retrieved {len(df)} rows × {len(df.columns)} cols in {elapsed:.2f}s")
        print(df.info())
    return df

def to(dataframe, destination_table, **kwargs):
    "Write DataFrame to BigQuery table"
    return _original_to_gbq(dataframe, destination_table, **kwargs)

def ex(query, project_id='petsathome-sb-datascience', **kwargs):
    "Execute query in BigQuery without returning results (useful for DDL)"
    client = bigquery.Client(project=project_id)
    job = client.query(query, **kwargs)
    return job.result()

def convert_bq_dtypes(df, date_cols=None, str_cols=None):
    "Convert BigQuery data types to pandas-compatible types"
    df = df.copy()
    date_cols = set(date_cols or [])
    str_cols = set(str_cols or [])
    for col in df.columns:
        if col in str_cols: df[col] = df[col].astype('object')
        elif re.search(r"(date|timestamp)", col.lower()) or col in date_cols: df[col] = pd.to_datetime(df[col], errors='coerce')
        elif df[col].dtype == 'float64': df[col] = df[col].astype('Float64')
        elif df[col].dtype == 'int64': df[col] = df[col].astype('Int64')
        elif df[col].dtype == 'object':
            first_val = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
            if first_val is not None and isinstance(first_val, Decimal): df[col] = df[col].astype('Float64')
    return df

```

#| hide
fastaistyle guide: 

Title: fastai

URL Source: https://docs.fast.ai/dev/style.html

Published Time: Fri, 14 Nov 2025 18:13:52 GMT

Markdown Content:
fastai coding style
-------------------

This is a brief discussion of fastai’s coding style, which is loosely informed by (a much diluted version of) the ideas developed over the last 60 continuous years of development in the [APL](https://en.wikipedia.org/wiki/APL_(programming_language)) / [J](https://en.wikipedia.org/wiki/J_(programming_language)) / [K](https://en.wikipedia.org/wiki/K_(programming_language)) programming communities, along with Jeremy’s personal experience contributing to programming language design and library development over the last 25 years. The style is particularly designed to be aligned with the needs of scientific programming and iterative, experimental development.

Everyone has strong opinions about coding style, except perhaps some very experienced coders, who have used many languages, who realize there’s lots of different perfectly acceptable approaches. The python community has particularly strongly held views, on the whole. I suspect this is related to Python being a language targeted at beginners, and therefore there are a lot of users with limited experience in other languages; however this is just a guess. Anyway, I don’t much mind what coding style you use when contributing to fastai, as long as:

*   You don’t change existing code to reduce its compliance with this style guide (especially: don’t use an automatic linter / formatter!)
*   You make some basic attempt to make your code not wildly different from the code that surrounds it.

Having said that, I do hope that you find the ideas in this style guide at least a little thought provoking, and that you consider adopting them to some extent when contributing to this library.

My personal approach to coding style is informed heavily by [Iverson’s](https://en.wikipedia.org/wiki/Kenneth_E._Iverson) Turing Award (the “Nobel of computer science”) lecture of 1979, [Notation as a Tool For Thought](http://www.eecg.toronto.edu/~jzhu/csc326/readings/iverson.pdf). If you can find the time, the paper is well worth reading and digesting carefully (it’s one of the most important papers in computer science history), representing the development of an idea that found its first expression in the release of APL in 1964. Iverson says:

> The thesis of the present paper is that the advantages of executability and universality found in programming languages can be effectively combined, in a single coherent language, with the advantages offered by mathematical notation

One key idea in the paper is that “_brevity facilitates reasoning_”, which has been incorporated into various guidelines such as “[shorten lines of communication](http://archive.vector.org.uk/art10009750)”. This is sometimes incorrectly assumed to just mean ‘terseness’, but it is a much deeper idea, as described in [this Hacker News thread](https://news.ycombinator.com/item?id=13595729). I can’t hope to summarize this thinking here, but I can point out a couple of key benefits:

*   It supports [expository programming](http://archive.vector.org.uk/art10000980), particularly when combined with the use of Jupyter Notebook or a similar tool designed for experimentation
*   The most productive programmers I’m aware of in the world, such as the extraordinary [Arthur Whitney](https://en.wikipedia.org/wiki/Arthur_Whitney_(computer_scientist)) often use this coding style (which may or may not be a coincidence!)

Style guide
-----------

Python has over time incorporated a number of ideas that make it more amenable to this form of programming, such as:

*   List, dictionary, generator, and set comprehensions
*   Lambda functions
*   Python 3.6 interpolated format strings
*   Numpy array-based programming.

Although Python will always be more verbose than many languages, by using these features liberally, along with some simple rules of thumb, we can aim to keep all the key ideas for one semantic concept in a single screen on code. This is one of my main goals when programming—I find it very hard to understand a concept if I have to jump around the place to put the bits together. (Or as Arthur Whitney says “I hate scrolling”!)

### Symbol naming

*   Follow standard Python casing guidelines (CamelCase classes, under_score for most everything else).
*   In general, aim for what Perl designer [Larry Wall](https://developers.slashdot.org/story/16/07/14/1349207/the-slashdot-interview-with-larry-wall) describes as metaphorically as _Huffman Coding_:

> In metaphorical honor of Huffman’s compression code that assigns smaller numbers of bits to more common bytes. In terms of syntax, it simply means that commonly used things should be shorter, but you shouldn’t waste short sequences on less common constructs.

*   A fairly complete list of abbreviations is in [abbr.qmd](https://docs.fast.ai/dev/abbr.html); if you see anything missing, feel free to edit this file.
*   For example, that in computer vision code, where we say ‘size’ and ‘image’ all the time, we use shortened forms `sz` and `img`. Or in NLP code, we would say `lm` instead of ‘language model’
*   Use `o` for an object in a comprehension, `i` for an index, and `k` and `v` for a key and value in a dictionary comprehension.
*   Use `x` for a tensor input to an algorithm (e.g.layer, transform, etc), unless interoperating with a library where this isn’t the expected behavior (e.g.if writing a pytorch loss function, use `input` and `target` as is standard for that library).
*   Take a look at the naming conventions in the part of code you’re working on, and try to stick with them. E.g. in `fastai.transforms` you’ll see ‘det’ for ‘deterministic’, ‘tfm’ for ‘transform’, and ‘coord’ for coordinate.
*   Assume the coder has knowledge of the domain in which you’re working 
    *   For instance, use `kl_divergence` not `kullback_leibler_divergence`; or (like pytorch) use `nll` not `negative_log_likelihood`. If the coder doesn’t know these terms, they will need to look them up in the docs anyway and learn the concepts; if they do know the terms, the abbreviations will be well understood
    *   When implementing a paper, aim to follow the paper’s nomenclature, unless it’s inconsistent with other widely-used conventions. E.g. `conv1` not `first_convolutional_layer`

Although it’s hard to design a really compelling experiment for this kind of thing, there is some [interesting research](https://www.sciencedirect.com/science/article/pii/S0167642309000343) supporting the idea that overly long symbol names negatively impact code comprehension.

### Layout

*   Code should be less wide than the number of characters that fill a standard modern small-ish screen (currently 1600x1200) at 14pt font size. That means around 160 characters. Following this rule will mean very few people will need to scroll sideways to see your code. (If they’re using a jupyter notebook theme that restricts their cell width, that’s on them to fix!)

*   One line of code should implement one complete idea, where possible

*   Generally therefore an `if` part and its 1-line statement should be on one line, using `:` to separate

*   Using the ternary operator `x = y if a else b` can help with this guideline

*   If a 1-line function body comfortably fits on the same line as the `def` section, feel free to put them together with `:`

*   If you’ve got a bunch of 1-line functions doing similar things, they don’t need a blank line between them

```
def det_lighting(b, c): return lambda x: lighting(x, b, c)
def det_rotate(deg): return lambda x: rotate_cv(x, deg)
def det_zoom(zoom): return lambda x: zoom_cv(x, zoom)
``` 
*   Aim to align statement parts that are conceptually similar. It allows the reader to quickly see how they’re different. E.g. in this code it’s immediately clear that the two parts call the same code with different parameter orders.

```
if self.store.stretch_dir==0: x = stretch_cv(x, self.store.stretch, 0)
else:                         x = stretch_cv(x, 0, self.store.stretch)
``` 
*   Put all your class member initializers together using destructuring assignment. When doing so, use no spaces after the commas, but spaces around the equals sign, so that it’s obvious where the LHS and RHS are.

`self.sz,self.denorm,self.norm,self.sz_y = sz,denorm,normalizer,sz_y` 
*   Avoid using vertical space when possible, since vertical space means you can’t see everything at a glance. For instance, prefer importing multiple modules on one line.

`import PIL, os, numpy as np, math, collections, threading` 
*   Indent with 4 spaces. (In hindsight I wish I’d picked 2 spaces, like Google’s style guide, but I don’t feel like going back and changing everything…)

*   When it comes to adding spaces around operators, try to follow notational conventions such that your code looks similar to domain specific notation. E.g. if using pathlib, don’t add spaces around `/` since that’s not how we write paths in a shell. In an equation, use spacing to lay out the separate parts of an equation so it’s as similar to regular math layout as you can.

*   Avoid trailing whitespace

### Algorithms

*   fastai is designed to show off the best of what’s possible. So try to ensure that your implementation of an algorithm is at least as fast, accurate, and concise as other versions that exist (if they do), and use a profiler to check for hotspots and optimize them as appropriate (if the code takes more than a second to run in practice).
*   Try to ensure that your algorithm scales nicely; specifically, it should work in 16GB RAM on arbitrarily large datasets. That will generally mean using lazy data structures such as generators, and not pulling everything in to a list.
*   Add a comment that provides the equation number from the paper that you’re implementing in the appropriate part of the code.
*   Use numpy/pytorch broadcasting, not loops, where possible.
*   Use numpy/pytorch advanced indexing, not specialized indexing methods, where possible.
*   Don’t submit a PR that implements the latest hot paper until you’ve actually tried using it on a few datasets, compared it to existing approaches, and confirmed it’s actually useful in practice! Ideally, include a notebook as a gist link with your PR showing these results.

### Other stuff

*   Feel free to assume the latest version of python and key libraries is installed. But do mention in the PR and docs if you’re relying on something that’s only a couple of months old (including recently fixed bugs). Don’t rely on any unreleased or beta versions however.
*   Avoid comments unless they are necessary to tell the reader _why_ you’re doing something. To tell them _how_ you’re doing it, use symbol names and clear expository code.
*   If you’re implementing a paper or following some other external document, include a link to it in your code.
*   If you’re using nearly all the stuff provided by a module, just `import *`. There’s no need to list all the things you are importing separately! To avoid exporting things which are really meant for internal use, define [`__all__`](https://stackoverflow.com/questions/44834/can-someone-explain-all-in-python). currently following the `__all__` guideline, and welcome PRs to fix this.)
*   Assume the user has a modern editor or IDE and knows how to use it. E.g. if they want to browse the methods and classes, they can use code folding - they don’t need to rely on having two lines between classes. If they want to see the definition of a symbol they can jump to the reference/tag, then don’t need a list of imports at the top of the file. And so forth…
*   Don’t use an automatic linter like autopep8 or formatter like yapf. No automatic tool can lay out your code with the care and domain understanding that you can. And it’ll break all the care and domain understanding that previous contributors have used in that file!
*   Keep your PRs small, and for anything controversial or tricky discuss it on [the forums](https://forums.fast.ai/) first.

### Documentation

*   Documentation largely goes in the notebooks in `docs_src`, which is used to create the HTML docs
*   In the code, add a one-line docstring which includes back-quoted references to the main params by name
*   The [Python re module](https://docs.python.org/3/library/re.html) is a good role model for the documentation style we’re looking for.

FAQ
---

 Why not use PEP 8?  I don’t think it’s ideal for the style of programming that we use, or for math-heavy code. If you’ve never used anything except PEP 8, here’s a chance to experiment and learn something new!  My editor is complaining about PEP 8 violations in fastai; what should I do?  Pretty much all editors have the ability to disable linting for a project; figure out how to do that in your editor.  Are you worried that using a different style guide might put off new contributors?  Not really. We’re really not that fussy about style, so we won’t be rejecting PRs that aren’t formatted according to this document. And whilst there are people around who are so closed-minded that they can’t handle new things, they’re certainly not the kind of people we want to be working with!

# core

> Ergonomic wrapper for pandas_gbq that simplifies loading BigQuery data into DataFrames

In [ ]:
#| default_exp core

In [ ]:


#| export
from pandas_gbq import read_gbq as _original_read_gbq, to_gbq as _original_to_gbq, Context, context
import pandas as pd, re, time
from decimal import Decimal
from google.cloud import bigquery


In [ ]:
#| hide
from nbdev.showdoc import *
from google.oauth2 import service_account
import os
import json



The `read` function wraps `pandas_gbq.read_gbq` with convenient defaults and optional type conversion.


In [ ]:
#| export
def read(
    query_or_table:str, # SQL query string or table reference
    verbose:bool=True, # Print timing and DataFrame info
    convert_dtypes:bool=True, # Apply type conversion to columns
    date_cols:list=None, # List of columns to convert to datetime
    str_cols:list=["scv_id"], # List of columns to keep as string/object type
    use_bqstorage_api:bool=True, # Use BigQuery Storage API for faster reads
    **kwargs
    ):
    start = time.time()
    df = _original_read_gbq(query_or_table, use_bqstorage_api=use_bqstorage_api, **kwargs)
    if convert_dtypes: df = convert_bq_dtypes(df, date_cols=date_cols, str_cols=str_cols)
    if verbose:
        elapsed = time.time() - start
        print(f"Retrieved {len(df)} rows × {len(df.columns)} cols in {elapsed:.2f}s")
        print(df.info())
        try:
            from IPython.display import display
            display(df.head())
        except ImportError: print(df.head())
    return df

def to(dataframe, destination_table, **kwargs):
    "Write DataFrame to BigQuery table"
    return _original_to_gbq(dataframe, destination_table, **kwargs)

def ex(query, project_id='petsathome-sb-datascience', **kwargs):
    "Execute query in BigQuery without returning results (useful for DDL)"
    client = bigquery.Client(project=project_id)
    job = client.query(query, **kwargs)
    return job.result()

def convert_bq_dtypes(df, date_cols=None, str_cols=None):
    "Convert BigQuery data types to pandas-compatible types"
    df = df.copy()
    date_cols = set(date_cols or [])
    str_cols = set(str_cols or [])
    for col in df.columns:
        if col in str_cols: df[col] = df[col].astype('object')
        elif re.search(r"(date|timestamp)", col.lower()) or col in date_cols: df[col] = pd.to_datetime(df[col], errors='coerce')
        elif df[col].dtype == 'float64': df[col] = df[col].astype('Float64')
        elif df[col].dtype == 'int64': df[col] = df[col].astype('Int64')
        elif df[col].dtype == 'object':
            first_val = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
            if first_val is not None and isinstance(first_val, Decimal): df[col] = df[col].astype('Float64')
    return df



How can i check my fastcore docs worked by previewing my nbdev docs?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You can preview your nbdev docs locally by running:

```python
!nbdev_preview
```

This will start a local server and show you the URL where you can view your documentation. It watches for changes and automatically rebuilds as you modify your notebooks.

Alternatively, if you want to just build the docs without serving them:

```python
!nbdev_docs
```

Then you can open the HTML files in the `_docs` directory directly in your browser.

Hmm is there an easier way to do it from within a notebook? check the docs and let me knwo? https://nbdev.fast.ai/tutorials/tutorial.html

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



<details class='tool-usage-details'>

```json
{
  "id": "toolu_01NjZmeNaygu5ZeHpeEdg9kU",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://nbdev.fast.ai/tutorials/tutorial.html"
    }
  },
  "result": "'[ nbdev ](../index.html)\\n\\n  * [ Get Started](../getting_started.html)\\n  * [ Tutorial](../tutoria<TRUNCATED>"
}
```

</details>

Yes! You can preview your docs locally by running `nbdev_preview` in the terminal, which uses Quarto to render your docs on your computer and keep them updated as you edit your notebooks.

Here's the code to run in a notebook cell:

```python
!nbdev_preview
```

It will display something like "Watching files for changes" and "Browse at http://localhost:3000/" - then you can click the link to open the preview in a new browser tab.

In [ ]:
gn_query = """
-- top 10 girls names
SELECT name, sum(number) as name_count
 FROM `bigquery-public-data.usa_names.usa_1910_current` 
 where year = 2021
  and gender = 'F'
 group by all
 order by name_count desc
 LIMIT 10
"""

gn_query[:20]

'\n-- top 10 girls nam'

In [ ]:

credentials = service_account.Credentials.from_service_account_info(
    json.loads(os.environ['GOOGLE_APPLICATION_CREDENTIALS'])
)
credentials


<google.oauth2.service_account.Credentials>

In [ ]:

df = read(gn_query, credentials=credentials, project_id='bq-sandbox-motdam', convert_dtypes=False)


Downloading:   0%|          |

Downloading: 100%|██████████|


Retrieved 10 rows × 2 cols in 0.80s
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        10 non-null     object
 1   name_count  10 non-null     Int64 
dtypes: Int64(1), object(1)
memory usage: 302.0+ bytes
None


,name,name_count
0,Olivia,17728
1,Emma,15433
2,Charlotte,13285
3,Amelia,12952
4,Ava,12759


I am trying to create a nbdev package with a wrapper for the pandas pandas_gbq lib, can you please help me create my core.py nb? I want it to use an interactive fastai style source code as documentation approach that clearly lays out how the library works



##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I'd be happy to help you create an nbdev notebook that documents your pandas_gbq wrapper! 

Looking at what you've already built, you have some nice functionality. To make this really shine with the fastai interactive documentation style, let's think about the best way to structure it.

The fastai approach typically:
1. Introduces each function with a clear example showing it in action
2. Explains parameters as they become relevant through examples
3. Builds up complexity gradually

Before I suggest specific additions, I'd like to understand your vision better:

1. What aspects of the current notebook would you like to improve or reorganize?
2. Do you want to keep the example with the girls' names query, or would you prefer a different example that might be more representative of typical usage?
3. Are there other functions or features you're planning to add beyond `read`, `to`, `ex`, and `convert_bq_dtypes`?

What would be most helpful to focus on first?

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

Hoq can i get a previoe of what the docs look like so far?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You can preview your docs by running:

```python
!nbdev_preview
```

This will start a local server and give you a URL (usually http://localhost:3000/) where you can see your rendered documentation with live updates as you edit.

Since I'm in sovleit is that going to work? i though solve used port 8000 by default

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right to be concerned about port conflicts. In solveit, you can specify a different port for `nbdev_preview`:

```python
!nbdev_preview --port 3001
```

Or you could use any other available port. The command will show you the URL to access once it starts up.

Okay i ran that in a terminal cell, what now?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Now you should see output showing that the preview server is running and watching for changes. Look for a line that says something like "Browse at http://localhost:3001/" or similar.

You can click on that URL (or copy/paste it into a new browser tab) to view your rendered documentation. As you make changes to your notebook and save, the preview will automatically update.

If you're not seeing the URL or the server isn't starting, what output are you seeing from the command?

I can' tlok at local host though because i'm in solveit, i think i have to do something specific like publish the notebook or the folder, or use the preview() fucniton in fasthtml, can you figoure out for me?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Ah yes, you're right! In solveit you need to use a different approach. Let me check the nbdev docs for the best way to handle this.For solveit, you can't access localhost directly. Instead, you need to build the docs to HTML files and then serve them. Try this:

```python
!nbdev_docs
```

This will build your docs to the `_docs` folder. Then you can view the HTML files directly by opening them in your browser, or you could use Python's built-in HTTP server on a different port:

```python
!cd _docs && python -m http.server 8001
```

Then access it through solveit's interface at the appropriate URL for port 8001.

SHodl i worry about these errors?
```
bash: !nbdev_preview: event not found
solveit@f6037f604d7f:~/bqdf/nbs$ nbdev_preview --port 3001
Terminating existing preview server....DONE

Preparing to preview
[1/1] 00_core.ipynb

Watching files for changes
Browse at http://localhost:3001/core.html
Listening on http://127.0.0.1:3001/
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/nbdev/quarto.py", line 367, in _f
    try: serve_drv.main(res)
         ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/nbdev/serve_drv.py", line 22, in main
    if src.suffix=='.ipynb': exec_nb(src, dst, x)
                             ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/nbdev/serve_drv.py", line 15, in exec_nb
    nb = read_nb(src)
         ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/execnb/nbio.py", line 59, in read_nb
    res = dict2nb(_read_json(path, encoding='utf-8'))
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/execnb/nbio.py", line 18, in _read_json
    return loads(Path(self).read_text(encoding=encoding, errors=errors))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/json/decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/json/decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
pandoc 
  to: html
  output-file: /app/data/bqdf/_proc/core.html
  standalone: true
  title-prefix: bqdf
  section-divs: true
  html-math-method: mathjax
  wrap: none
  default-image-extension: png
  css:
    - styles.css
  toc: true
  variables: {}
  
metadata
  document-css: false
  link-citations: true
  date-format: long
  lang: en
  theme: cosmo
  description: Ergonomic wrapper for pandas_gbq that simplifies loading BigQuery data into DataFrames
  title: core
  
pandoc 
  to: html
  output-file: /app/data/bqdf/_proc/core.html
  standalone: true
  title-prefix: bqdf
  section-divs: true
  html-math-method: mathjax
  wrap: none
  default-image-extension: png
  css:
    - styles.css
  toc: true
  variables: {}
  
metadata
  document-css: false
  link-citations: true
  date-format: long
  lang: en
  theme: cosmo
  description: Ergonomic wrapper for pandas_gbq that simplifies loading BigQuery data into DataFrames
  title: core
  
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/nbdev/quarto.py", line 367, in _f
    try: serve_drv.main(res)
         ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/nbdev/serve_drv.py", line 22, in main
    if src.suffix=='.ipynb': exec_nb(src, dst, x)
                             ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/nbdev/serve_drv.py", line 15, in exec_nb
    nb = read_nb(src)
         ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/execnb/nbio.py", line 59, in read_nb
    res = dict2nb(_read_json(path, encoding='utf-8'))
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/execnb/nbio.py", line 18, in _read_json
    return loads(Path(self).read_text(encoding=encoding, errors=errors))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/json/decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/json/decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/nbdev/quarto.py", line 367, in _f
    try: serve_drv.main(res)
         ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/nbdev/serve_drv.py", line 22, in main
    if src.suffix=='.ipynb': exec_nb(src, dst, x)
                             ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/nbdev/serve_drv.py", line 15, in exec_nb
    nb = read_nb(src)
         ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/execnb/nbio.py", line 59, in read_nb
    res = dict2nb(_read_json(path, encoding='utf-8'))
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/execnb/nbio.py", line 18, in _read_json
    return loads(Path(self).read_text(encoding=encoding, errors=errors))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/json/decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/json/decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
pandoc 
  to: html
  output-file: /app/data/bqdf/_proc/core.html
  standalone: true
  title-prefix: bqdf
  section-divs: true
  html-math-method: mathjax
  wrap: none
  default-image-extension: png
  css:
    - styles.css
  toc: true
  variables: {}
  
metadata
  document-css: false
  link-citations: true
  date-format: long
  lang: en
  theme: cosmo
  description: Ergonomic wrapper for pandas_gbq that simplifies loading BigQuery data into DataFrames
  title: core
  ```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The JSON decode errors suggest there's an empty or corrupted notebook file somewhere. The first bash error is just because you used `!` in the terminal (that's for notebook cells, not terminal).

The preview server is actually running and watching for changes. However, since you're in solveit, you won't be able to access `localhost:3001` directly.

Instead, let's just build the static docs and view them:

```python
!nbdev_docs
```

Then you can open the HTML files in `_docs/` directly, or serve them with:

```python
from pathlib import Path
from starlette.applications import Starlette
from starlette.routing import Mount
from starlette.staticfiles import StaticFiles

app = Starlette(routes=[Mount('/', StaticFiles(directory='_docs', html=True), name='docs')])
```

Then serve it on port 8001 or similar so you can access it through solveit's port forwarding.

In [ ]:
from pathlib import Path
from starlette.applications import Starlette
from starlette.routing import Mount
from starlette.staticfiles import StaticFiles

app = Starlette(routes=[Mount('/', StaticFiles(directory='_docs', html=True), name='docs')])


RuntimeError: Directory '_docs' does not exist

hmmm whats the issue here

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You need to build the docs first. Run this:

```python
!nbdev_docs
```

I have!

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check what directory you're in and where the `_docs` folder actually is:

```python
!pwd
!ls -la
```

In [ ]:
!pwd
!ls -la


/app/data/bqdf/nbs


total 144
drwxrwsr-x 2 solveit solveit  4096 Nov 23 12:13 .
drwxrwsr-x 8 solveit solveit  4096 Nov 23 12:24 ..
-rw-rw-r-- 1 solveit solveit 51357 Nov 23 12:25 00_core.ipynb
-rw-rw-r-- 1 solveit solveit 63027 Nov 23 11:59 00_core_scratch.ipynb
-rw-rw-r-- 1 solveit solveit   330 Nov 23 09:14 _quarto.yml
-rw-rw-r-- 1 solveit solveit  3800 Nov 23 09:14 index.ipynb
-rw-rw-r-- 1 solveit solveit   268 Nov 23 12:24 nbdev.yml
-rw-rw-r-- 1 solveit solveit   106 Nov 23 12:24 sidebar.yml
-rw-rw-r-- 1 solveit solveit   600 Nov 23 09:14 styles.css


Sure thats done, now how do i see it? also what is starlette?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Starlette is a lightweight ASGI framework (it's what FastAPI is built on). Now you need to serve the app:

```python
import uvicorn
uvicorn.run(app, host='0.0.0.0', port=8001)
```

This will start a server on port 8001 that serves your docs. In solveit, you should be able to access it through the port forwarding interface.